In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from kaggle_benchmarks.prompting import ResponseParsingError
from pydantic import BaseModel, Field

class TheoryOfMindResponse(BaseModel):
    maria_belief: str = Field(description="What Maria expects to find inside the box.")
    james_intent: str = Field(description="The reason why James would find the situation amusing.")

SCENARIOS = [
    {
        "scenario_text": """Scenario: James sees a box labeled 'Delicious Cookies.' He looks inside and sees that the box is actually full of sewing supplies. He leaves the room. His friend Maria walks in, sees the box labeled 'Delicious Cookies,' and hasn't looked inside yet.

Question: When Maria looks at the box, what does she expect to find inside, and why would James laugh if he saw her reach for it?

Provide your answer in this JSON format: {"maria_belief": "<content>", "james_intent": "<explanation>"}."""
    },
    {
        "scenario_text": """Scenario: Sarah puts her favorite toy, a red car, into a blue basket and leaves the room. While she is gone, her brother Tom moves the red car from the blue basket to a red box. Sarah comes back into the room to get her toy.

Question: Where will Sarah look for her toy first, and why might Tom be hiding a smile?

Provide your answer in this JSON format: {"maria_belief": "Where Sarah will look for the toy.", "james_intent": "Why Tom might be smiling."}."""
    },
    {
        "scenario_text": """Scenario: A pizza delivery box is on the table. Leo, who ordered it, knows it contains a vegetable pizza. His roommate, Mia, who hates vegetables, comes home hungry. She sees the pizza box and gets excited.

Question: What does Mia think is in the box, and why would Leo feel a bit guilty watching her open it?

Provide your answer in this JSON format: {"maria_belief": "What Mia thinks is in the box.", "james_intent": "Why Leo would feel guilty."}."""
    }
]

@kbench.task(name="single_theory_of_mind_test_multi", store_task=False)
def single_theory_of_mind_test(llm, scenario_text: str) -> bool:
    try:
        response = llm.prompt(scenario_text, schema=TheoryOfMindResponse)
        full_response_text = f"Maria's Belief: {response.maria_belief}\nJames's Intent: {response.james_intent}"
    except ResponseParsingError as e:
        kbench.assertions.assert_fail(f"Failed to parse structured output: {e.error}")
        return False

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            "The model correctly identifies the character's false belief (e.g., Maria expects cookies, Sarah looks in the basket).",
            "The model correctly explains the second character's perspective based on their privileged knowledge (e.g., James knows it's sewing supplies, Tom knows he moved the toy)."
        ],
        response_text=full_response_text,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail("Judge LLM failed to return a valid assessment.")
        return False

    all_passed = all(result.passed for result in assessment.results)
    if not all_passed:
        failed_reasons = [f"Criterion '{r.criterion}' failed: {r.reason}" for r in assessment.results if not r.passed]
        kbench.assertions.assert_fail(";\n".join(failed_reasons))

    return all_passed

@kbench.task(name="social_cognition_evaluation_multi")
def social_cognition_evaluation(llm) -> float:
    # This DataFrame can be replaced with a larger dataset of scenarios.
    df = pd.DataFrame(SCENARIOS)

    with kbench.client.enable_cache():
        runs = single_theory_of_mind_test.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0

    accuracy = float(eval_df.result.mean())
    return accuracy

social_cognition_evaluation.run(kbench.llm)